# Worldwide Earthquake Events API - Silver Layer Processing

In [3]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

StatementMeta(, d19b26bf-8998-4efa-ba33-f83e24621794, 5, Finished, Available, Finished, False)

In [4]:
# To be commented out later
start_date = date.today() - timedelta(7)
end_date = date.today() - timedelta(1)

# df now is a Spark DataFrame containing JSON data
df = spark.read.option("multiline", "true").json(f"Files/{start_date}_earthquake_data.json")

StatementMeta(, d19b26bf-8998-4efa-ba33-f83e24621794, 6, Finished, Available, Finished, False)

In [5]:
# Reshape earthquake data by extracting and renaming key attributes for further analysis.
df = \
df.\
    select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
        )

StatementMeta(, d19b26bf-8998-4efa-ba33-f83e24621794, 7, Finished, Available, Finished, False)

In [6]:
# Convert 'time' and 'updated' columns from milliseconds to timestamp format for clearer datetime representation.
df = df.\
    withColumn('time', col('time')/1000).\
    withColumn('updated', col('updated')/1000).\
    withColumn('time', col('time').cast(TimestampType())).\
    withColumn('updated', col('updated').cast(TimestampType()))

StatementMeta(, d19b26bf-8998-4efa-ba33-f83e24621794, 8, Finished, Available, Finished, False)

In [7]:
# appending the data to the gold table
df.write.mode('append').saveAsTable('earthquake_events_silver')

StatementMeta(, d19b26bf-8998-4efa-ba33-f83e24621794, 9, Finished, Available, Finished, False)